# 📊 Analisis Probabilitas PR Membutuhkan Lebih dari 14 Hari untuk Di-merge

Notebook ini menganalisis dataset Pull Request (PR) untuk menjawab pertanyaan:
> **Berapa probabilitas sebuah PR baru akan membutuhkan waktu lebih dari 14 hari untuk berhasil di-merge?**

---


In [ ]:
# ============================================================
# 1. IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil di-import")


In [ ]:
# ============================================================
# 2. LOAD & PREVIEW DATASET
# ============================================================
df = pd.read_csv('dataset_prs_clean.csv')

print(f"Ukuran dataset : {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"Kolom          : {df.columns.tolist()}")
print()
df.head(5)


In [ ]:
# ============================================================
# 3. EKSPLORASI DATA (EDA)
# ============================================================
print("=== Distribusi Status Merge ===")
print(df['status_merge'].value_counts())
print()
print("=== Distribusi State PR ===")
print(df['state'].value_counts())
print()
print("=== Info Tipe Data & Missing Values ===")
print(df.dtypes)
print()
print("Missing values per kolom:")
print(df.isnull().sum())


In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================
# Parse datetime
df['created_at'] = pd.to_datetime(df['created_at'], utc=True)
df['merged_at']  = pd.to_datetime(df['merged_at'],  utc=True)
df['closed_at']  = pd.to_datetime(df['closed_at'],  utc=True)

# Hitung durasi (hari) sejak dibuat sampai di-merge (hanya PR yg merged)
df['time_to_merge_days'] = np.where(
    df['is_merged'] == 1,
    (df['merged_at'] - df['created_at']).dt.total_seconds() / 86400,
    np.nan
)

# Label biner: apakah PR butuh > 14 hari untuk di-merge?
df['over_14_days'] = np.where(
    df['is_merged'] == 1,
    (df['time_to_merge_days'] > 14).astype(int),
    np.nan   # PR yang belum/tidak merged dibiarkan NaN
)

# Kategori durasi
def categorize_duration(days):
    if pd.isna(days):
        return 'Not Merged'
    elif days <= 1:
        return '0-1 hari'
    elif days <= 7:
        return '2-7 hari'
    elif days <= 14:
        return '8-14 hari'
    elif days <= 30:
        return '15-30 hari'
    else:
        return '> 30 hari'

df['duration_category'] = df['time_to_merge_days'].apply(categorize_duration)

print("✅ Feature engineering selesai")
print(df[['number','status_merge','time_to_merge_days','over_14_days','duration_category']].head(8).to_string())


In [ ]:
# ============================================================
# 5. PERHITUNGAN PROBABILITAS
# ============================================================
merged_df   = df[df['is_merged'] == 1].copy()
total_all   = len(df)
total_merged = len(merged_df)

over_14_count = (merged_df['time_to_merge_days'] > 14).sum()

# --- Probabilitas utama (kondisional: diberikan PR akhirnya merged) ---
P_over14_given_merged = over_14_count / total_merged

# --- Probabilitas marginal (dari semua PR yang masuk) ---
P_over14_marginal = over_14_count / total_all

# --- Statistik deskriptif durasi ---
desc = merged_df['time_to_merge_days'].describe(percentiles=[.25,.5,.75,.90,.95])

print("=" * 55)
print("  HASIL PERHITUNGAN PROBABILITAS")
print("=" * 55)
print(f"  Total PR dalam dataset        : {total_all:,}")
print(f"  Total PR yang berhasil merged : {total_merged:,}")
print(f"  PR merged yg > 14 hari       : {over_14_count:,}")
print()
print(f"  ▶ P(> 14 hari | PR di-merge)  : {P_over14_given_merged:.4f}  ({P_over14_given_merged*100:.2f}%)")
print(f"  ▶ P(> 14 hari, marginal)      : {P_over14_marginal:.4f}  ({P_over14_marginal*100:.2f}%)")
print("=" * 55)
print()
print("--- Statistik Durasi (hari) untuk PR yang Merged ---")
print(desc.round(2).to_string())


In [ ]:
# ============================================================
# 6. VISUALISASI
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analisis Durasi Pull Request Hingga Di-merge', fontsize=15, fontweight='bold', y=1.01)

# --- Plot 1: Histogram durasi (capped 90 hari) ---
ax1 = axes[0, 0]
data_capped = merged_df['time_to_merge_days'].clip(upper=90)
ax1.hist(data_capped, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(14, color='crimson', linewidth=2, linestyle='--', label='Batas 14 hari')
ax1.set_title('Distribusi Durasi Merge (capped 90 hari)')
ax1.set_xlabel('Hari')
ax1.set_ylabel('Jumlah PR')
ax1.legend()

# --- Plot 2: Pie chart kategori durasi (merged only) ---
ax2 = axes[0, 1]
order  = ['0-1 hari', '2-7 hari', '8-14 hari', '15-30 hari', '> 30 hari']
colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
counts = merged_df['duration_category'].value_counts().reindex(order, fill_value=0)
ax2.pie(counts, labels=order, colors=colors, autopct='%1.1f%%', startangle=140)
ax2.set_title('Proporsi Kategori Durasi Merge')

# --- Plot 3: Bar chart > 14 vs <= 14 hari ---
ax3 = axes[1, 0]
labels_bar = ['≤ 14 hari', '> 14 hari']
vals   = [total_merged - over_14_count, over_14_count]
colors_bar = ['#27ae60', '#e74c3c']
bars = ax3.bar(labels_bar, vals, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:,}\n({val/total_merged*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax3.set_title('PR Merged: ≤ 14 hari vs > 14 hari')
ax3.set_ylabel('Jumlah PR')
ax3.set_ylim(0, max(vals) * 1.2)

# --- Plot 4: Probabilitas ringkasan ---
ax4 = axes[1, 1]
ax4.axis('off')
summary_text = (
    "RINGKASAN PROBABILITAS\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    f"Total PR          : {total_all:,}\n"
    f"PR Berhasil Merged: {total_merged:,}\n"
    f"PR > 14 hari      : {over_14_count:,}\n\n"
    f"P(>14 hari | merged)\n"
    f"  = {over_14_count}/{total_merged}\n"
    f"  = {P_over14_given_merged:.4f}\n"
    f"  ≈ {P_over14_given_merged*100:.2f}%\n\n"
    f"P(>14 hari, marginal)\n"
    f"  = {over_14_count}/{total_all}\n"
    f"  = {P_over14_marginal:.4f}\n"
    f"  ≈ {P_over14_marginal*100:.2f}%"
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('pr_analysis_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart tersimpan sebagai 'pr_analysis_chart.png'")


In [ ]:
# ============================================================
# 7. SIMPAN DATASET BARU
# ============================================================
output_path = 'dataset_prs_enriched.csv'
df.to_csv(output_path, index=False)

print(f"✅ Dataset baru tersimpan: '{output_path}'")
print(f"   Ukuran: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print()
print("Kolom baru yang ditambahkan:")
new_cols = ['time_to_merge_days', 'over_14_days', 'duration_category']
print(df[new_cols].describe(include='all').to_string())
print()
print("Sampel dataset baru:")
df[['number','status_merge','time_to_merge_days','over_14_days','duration_category']].head(10)


In [ ]:
# ============================================================
# 8. KESIMPULAN
# ============================================================
print("=" * 60)
print("  KESIMPULAN ANALISIS")
print("=" * 60)
print()
print(f"  Dari {total_all:,} PR yang dianalisis:")
print()
print(f"  1. PROBABILITAS KONDISIONAL")
print(f"     Jika sebuah PR PASTI akan di-merge, maka")
print(f"     peluang prosesnya memakan waktu > 14 hari adalah:")
print()
print(f"     P(> 14 hari | PR merged) = {P_over14_given_merged*100:.2f}%")
print()
print(f"  2. PROBABILITAS MARGINAL")  
print(f"     Dari seluruh PR yang masuk (merged maupun tidak),")
print(f"     peluang sebuah PR baru akhirnya merged dalam > 14 hari:")
print()
print(f"     P(> 14 hari) = {P_over14_marginal*100:.2f}%")
print()
print(f"  3. INTERPRETASI")
print(f"     - Median durasi merge : {merged_df['time_to_merge_days'].median():.1f} hari")
print(f"     - Mean durasi merge   : {merged_df['time_to_merge_days'].mean():.1f} hari")
print(f"     - Sebagian besar PR ({((merged_df['time_to_merge_days'] <= 7).sum()/total_merged*100):.1f}%)")
print(f"       berhasil di-merge dalam 7 hari pertama.")
print("=" * 60)
